# Chapter 2: LLMs and Evaluation Basics
Parts of an LLM app (single call → conversation → retrieval → tools → agent loop) and the basic ways to evaluate.

In [2]:
from common import ask, chat, embed, cosine, client, load_data, MODEL
import numpy as np

data = load_data()["ch2"]

## 1. Single call (one prompt in, one answer out)
`temperature=0` makes answers more repeatable.

In [ ]:
print(ask("Summarize in one line: " + data["summary_email"], system="You summarize emails.", temperature=0))

## 2. Conversation + message roles
The model has no memory. We **send the whole history every time**.
Roles: `system` (rules), `user` (request), `assistant` (model reply), `tool` (tool result, see section 7).

In [ ]:
history = data["chat_demo"]          # list of {"role": ..., "content": ...}
for m in history:
    print(f"{m['role']:10}: {m['content']}")

print("\nMODEL:", chat(history, system="You are a helpful assistant."))

## 3. Prompt template (parts joined with delimiters)

In [3]:
parts = data["prompt_template"]       # role, objective, instructions, reasoning, output_format, examples

def build_prompt(email):
    sections = []
    for name, text in parts.items():
        sections.append(f"### {name.upper()}\n{text}")          # ### = delimiter
    sections.append(f"### CONTEXT (customer email)\n<email>\n{email}\n</email>")
    return "\n\n".join(sections)

prompt = build_prompt(data["template_email"])
print(prompt)
print("\n--- MODEL REPLY ---")
print(ask(prompt))

### ROLE
You are a polite customer support assistant.

### OBJECTIVE
Write a short reply to the customer email. Success means the customer knows the next step.

### INSTRUCTIONS
Do not promise refunds. Do not invent policies. Keep it under 60 words.

### REASONING
Think about what the customer really needs before writing.

### OUTPUT_FORMAT
Plain text, 2-3 sentences.

### EXAMPLES
Email: 'Where is my invoice?'
Reply: 'Hello! You can download invoices from Settings > Billing. Let me know if you need more help.'

### CONTEXT (customer email)
<email>
Hi, I cannot reset my password and I need access today.
</email>

--- MODEL REPLY ---
Hello! Please try resetting your password using the 'Forgot Password' link on our login page. If you continue to experience issues, please reply with details, and we'll be happy to assist you further.


## 4. Retrieval type 1: Keyword search (BM25)
Finds documents that **share words** with the question.

In [4]:
from rank_bm25 import BM25Okapi

docs = data["past_emails"]
bm25 = BM25Okapi([d.lower().split() for d in docs])

def bm25_rank(query):
    scores = bm25.get_scores(query.lower().split())
    return list(np.argsort(scores)[::-1])          # best doc index first

for q in data["retrieval_queries"]:
    best = bm25_rank(q)[0]
    print(f"{q!r:25} -> {docs[best]}")

'i still cant log in'     -> I still can't log in to my account, the password reset email never arrives
'reset my login'          -> I still can't log in to my account, the password reset email never arrives
'get my receipt'          -> Where can I download last month's receipt?


## 5. Retrieval type 2: Embedding search
Text → numbers. Similar **meaning** → similar numbers (cosine similarity, -1 to 1).

In [5]:
doc_vectors = embed(docs)

def embedding_rank(query):
    q = embed([query])[0]
    scores = [cosine(q, v) for v in doc_vectors]
    return list(np.argsort(scores)[::-1])

for q in data["retrieval_queries"]:
    best = embedding_rank(q)[0]
    print(f"{q!r:25} -> {docs[best]}")

'i still cant log in'     -> Can't sign in after the update, it says wrong password
'reset my login'          -> Can't sign in after the update, it says wrong password
'get my receipt'          -> Where can I download last month's receipt?


## 6. Retrieval type 3: Hybrid search with RRF
RRF (Reciprocal Rank Fusion): a doc scores `1 / (60 + rank)` in each list; add the scores.

In [6]:
def rrf(*rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

for q in data["retrieval_queries"]:
    best = rrf(bm25_rank(q), embedding_rank(q))[0]
    print(f"{q!r:25} -> {docs[best]}")

'i still cant log in'     -> I still can't log in to my account, the password reset email never arrives
'reset my login'          -> I still can't log in to my account, the password reset email never arrives
'get my receipt'          -> Where can I download last month's receipt?


## 7. Tools + agent loop (ReAct style)
The model asks for a tool, we run it, send the result back, repeat. **Always cap the steps.**

In [7]:
from google.genai import types

def get_order_status(order_id: str):
    return data["orders"].get(order_id, {"error": "order not found"})

def get_return_window_days(item_type: str):
    return data["return_windows_days"].get(item_type, "unknown")

TOOLS = {"get_order_status": get_order_status, "get_return_window_days": get_return_window_days}

tool_spec = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="get_order_status", description="Get status and item type of an order",
        parameters={"type": "OBJECT", "properties": {"order_id": {"type": "STRING"}}, "required": ["order_id"]}),
    types.FunctionDeclaration(
        name="get_return_window_days", description="Get return window in days for an item type",
        parameters={"type": "OBJECT", "properties": {"item_type": {"type": "STRING"}}, "required": ["item_type"]}),
])
config = types.GenerateContentConfig(tools=[tool_spec], temperature=0)

In [8]:
contents = [types.Content(role="user", parts=[types.Part(text=data["agent_question"])])]

for step in range(5):                                    # maximum 5 steps
    resp = client().models.generate_content(model=MODEL, contents=contents, config=config)
    msg = resp.candidates[0].content
    calls = [p.function_call for p in msg.parts if p.function_call]

    if calls:                                            # model wants tools
        contents.append(msg)
        for call in calls:
            result = TOOLS[call.name](**call.args)
            print(f"step {step}: TOOL {call.name}({dict(call.args)}) -> {result}")
            contents.append(types.Content(role="user", parts=[
                types.Part.from_function_response(name=call.name, response={"result": result})]))
        continue                                         # ask the model again

    print(f"step {step}: FINAL ANSWER ->", resp.text)    # no tool call = done
    break

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 16h49m28.814016518s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.

## 8. Evaluation setup A: Absolute, reference-based
We know the correct answer, so compare (exact match).

In [ ]:
correct = 0
for item in data["extraction"]:
    out = ask(item["email"], system="Extract the date. Reply ONLY as YYYY-MM-DD.").strip()
    ok = out == item["expected"]
    correct += ok
    print(f"{item['expected']} vs {out}  {'OK' if ok else 'WRONG'}")
print(f"Accuracy: {correct}/{len(data['extraction'])}")

## 9. Evaluation setup B: Absolute, reference-free
No correct answer. Check **rules**: starts with `Sender:` and has exactly 3 bullets.

In [ ]:
def check_rules(summary):
    lines = [l for l in summary.strip().splitlines() if l.strip()]
    starts_ok = lines[0].startswith("Sender:")
    bullets = [l for l in lines if l.strip().startswith(("-", "*", "•"))]
    return {"starts_with_sender": starts_ok, "exactly_3_bullets": len(bullets) == 3}

print("Summary A:", check_rules(data["summary_a"]))
print("Summary B:", check_rules(data["summary_b"]))

## 10. Evaluation setup C: Comparative (pairwise)
Ask a judge which summary is better. We run **both orders** (A,B then B,A) to catch position bias.

In [ ]:
def pairwise(email, first, second):
    prompt = (f"Email:\n{email}\n\nSummary A:\n{first}\n\nSummary B:\n{second}\n\n"
              'Which summary is better (accurate, short, clear)? Reply JSON: {"winner": "A" or "B"}')
    return ask(prompt, as_json=True)["winner"]

r1 = pairwise(data["summary_email"], data["summary_a"], data["summary_b"])     # a first
r2 = pairwise(data["summary_email"], data["summary_b"], data["summary_a"])     # b first
winner1 = "summary_a" if r1 == "A" else "summary_b"
winner2 = "summary_b" if r2 == "A" else "summary_a"
print("Order 1 winner:", winner1)
print("Order 2 winner:", winner2)
print("Consistent (no position bias):", winner1 == winner2)